In [1]:
import sys
sys.path.append("/home/zanardi/Codes/ML/PyCOMET/pycomet/")

In [2]:
import numpy as np
import pycomet as pcm
import tensorflow as tf

from pyDOE import lhs

In [3]:
inp_space = {
  "n":    [ 1e18,  1e20],
  "Ar":   [ 1e-1,  9e-1],
  "Ar4s": [ 1e-1,  9e-1],
  "Ar4p": [ 1e-1,  9e-1],
  "Ar+":  [1e-20, 1e-20],
  "e-":   [ 1e-3,  1e-1]
}

scenarios = 10
test_cases = 2

In [4]:
inp_space_mat = np.vstack(list(inp_space.values())).T
inp_space_lbl = list(inp_space.keys())

In [ ]:
d = inp_space_mat.shape[1]
train = lhs(d, scenarios)
train

In [ ]:

# Sample initial conditions
# ===================================
def sample_inp_space():
  # Contruct design matrix (DM)
  d = inp_space.shape[1]
  s = scenarios * 2 + test_cases
  x = lhs(d, s)
  # Sorting w.r.t. norm
  i = np.linalg.norm(x, axis=-1).argsort()
  x = x[i]
  # Rescale DM
  x = x * (inp_space[1] - inp_space[0]) + inp_space[0]
  # Splitting sampled initial solutions
  if (test_cases > 0):
    indices = pcm.utils.stratified_sampling(np.arange(s), test_cases)
    samples = [tf.nest.map_structure(lambda z: z[i], x) for i in indices]
    samples = {
      "train": samples[0][0::2],
      "valid": samples[0][1::2],
      "test": samples[1]
    }
  else:
    samples = {
      "train": x[0::2],
      "valid": x[1::2]
    }
  # Returning sampled initial solutions
  return samples

In [ ]:
dim = {
	"inp": [(3,1),(4,2)],  # Input node attributes
	"out": [(3,1)]         # Output node attributes
}

# Layers
# -------------------------------------
latent_dim = 8

common = {
  'kernel_initializer': 'GlorotNormal',
  'regularizer': {
    'name': 'l1+l2',
    'l1': 1e-4,
    'l2': 1e-4
  },
  'dropout': {'rate': 1e-3},
  'layer_norm': True,
  'trainable': True
}

lays_propty = {

  'edge_encoder': {
    'lay_id': 'EdgeEncoder',
    'kwargs': {
      'layer_size': [2, 32, 32, latent_dim],
      'activation': ['sigmoid']*2 + ['linear'],
      "static_edge_index": "./data/edge_index.txt",
      "static_edge_attr": None,
      "name": 'edge_encoder',
      **common
    }
  },

  'node_encoder': {
    'lay_id': 'FFBlock',
    'kwargs': {
      'layer_size': [1, 32, 32, latent_dim],
      'activation': ['sigmoid']*2 + ['linear'],
      "name": 'node_encoder',
      **common
    }
  },

  'graphnet': [{
    'lay_id': 'GraphNet',
    'kwargs': {
      'layer_size': {
        "node": [latent_dim, 32, 32, latent_dim],
        "edge": [latent_dim, 32, 32, latent_dim]
      },
      'activation': {
        "node": ['tanh']*2 + ['linear'],
        "edge": ['tanh']*2 + ['linear']
      },
      "static_edge_index": "./data/edge_index.txt",
      "name": f'graphnet_{i+1}',
      **common
    }
  } for i in range(2)],

  'node_decoder': {
    'lay_id': 'FFBlock',
    'kwargs': {
      'layer_size': [latent_dim, 32, 32, 1],
      'activation': ['sigmoid']*2 + ['linear'],
      "name": 'node_decoder',
      **common
    }
  }

}


In [ ]:
model = EncodeProcessDecode(
    dim=dim,
    lays_propty=lays_propty
)

In [ ]:
inputs = []
for i in dim["inp"]:
    shape = tuple(tf.nest.flatten(i))
    inputs.append(tf.keras.Input(shape))
outputs = model.forward(inputs, training=False)
graph = tf.keras.Model(inputs, outputs)

In [ ]:
graph.summary()